# Exploratory Data Analysis — Uber Eats ATD Dataset

This notebook explores the delivery data for Mexico, covering the key dimensions needed to understand delivery time patterns and build a predictive model for **Actual Time of Delivery (ATD)**.

**Dataset:** `BC_A&A_with_ATD.csv` (~1 M delivery trips, March–April 2025)

**Columns:**  
`region`, `territory`, `country_name`, `workflow_uuid`, `driver_uuid`, `delivery_trip_uuid`,  
`courier_flow`, `restaurant_offered_timestamp_utc`, `order_final_state_timestamp_local`,  
`eater_request_timestamp_local`, `geo_archetype`, `merchant_surface`,  
`pickup_distance` (km), `dropoff_distance` (km), `ATD` (minutes)

---
**Sections**
1. Setup & Data Loading
2. Dataset Overview
3. Target Variable — ATD Distribution
4. Categorical Features
5. Numerical Features & Distances
6. Time-Based Patterns
7. Relationships & Correlations
8. Missing Values & Data Quality
9. Key Insights

---
## 1. Setup & Data Loading

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})

SEED = 42
DATA_PATH = '../data/raw/BC_A&A_with_ATD.csv'

In [ ]:
dtype_map = {
    'region': 'category',
    'territory': 'category',
    'country_name': 'category',
    'courier_flow': 'category',
    'geo_archetype': 'category',
    'merchant_surface': 'category',
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtype_map,
    parse_dates=[
        'restaurant_offered_timestamp_utc',
        'order_final_state_timestamp_local',
        'eater_request_timestamp_local',
    ],
    na_values=['\\N', 'NULL', 'None', ''],
)

print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head(3)

---
## 2. Dataset Overview

In [ ]:
df.info(memory_usage='deep')

In [ ]:
df.describe(include='all').T

In [ ]:
# Missing values summary
missing = (
    df.isnull()
    .sum()
    .rename('count')
    .to_frame()
    .assign(pct=lambda x: (x['count'] / len(df) * 100).round(2))
    .sort_values('count', ascending=False)
)
display(missing)

In [ ]:
print('Date range (restaurant_offered_timestamp_utc):')
print(f"  From : {df['restaurant_offered_timestamp_utc'].min()}")
print(f"  To   : {df['restaurant_offered_timestamp_utc'].max()}")
print(f"  Span : {(df['restaurant_offered_timestamp_utc'].max() - df['restaurant_offered_timestamp_utc'].min()).days} days")

---
## 3. Target Variable — ATD Distribution

**ATD** = minutes from `restaurant_offered_timestamp_utc` to `order_final_state_timestamp_local`.

In [ ]:
atd = df['ATD'].dropna()

stats = {
    'count':  f'{len(atd):,}',
    'mean':   f'{atd.mean():.1f} min',
    'median': f'{atd.median():.1f} min',
    'std':    f'{atd.std():.1f} min',
    'min':    f'{atd.min():.1f} min',
    'p25':    f'{atd.quantile(0.25):.1f} min',
    'p75':    f'{atd.quantile(0.75):.1f} min',
    'p95':    f'{atd.quantile(0.95):.1f} min',
    'p99':    f'{atd.quantile(0.99):.1f} min',
    'max':    f'{atd.max():.1f} min',
}
pd.Series(stats, name='ATD').to_frame()

In [ ]:
thresholds = [5, 30, 60, 90, 120, 180, 240]
rows_t = []
for t in thresholds:
    rows_t.append({
        'threshold (min)': t,
        'n_above': int((atd > t).sum()),
        'pct_above': f"{(atd > t).mean() * 100:.2f}%",
    })
display(pd.DataFrame(rows_t))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Full distribution (log y-axis)
axes[0].hist(atd, bins=100, color='steelblue', edgecolor='none')
axes[0].set_yscale('log')
axes[0].set_title('ATD — Full Distribution (log y)')
axes[0].set_xlabel('Minutes')

# Clipped at p99
p99 = atd.quantile(0.99)
axes[1].hist(atd[atd <= p99], bins=80, color='steelblue', edgecolor='none')
axes[1].axvline(atd.median(), color='red', linestyle='--', label=f'Median {atd.median():.0f}m')
axes[1].axvline(atd.mean(), color='orange', linestyle='--', label=f'Mean {atd.mean():.0f}m')
axes[1].legend()
axes[1].set_title(f'ATD — Clipped at p99 ({p99:.0f} min)')
axes[1].set_xlabel('Minutes')

# Box plot
axes[2].boxplot(atd[atd <= p99], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].set_title('ATD — Box Plot (≤ p99)')
axes[2].set_ylabel('Minutes')
axes[2].set_xticks([])

plt.suptitle('ATD (Actual Time of Delivery)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Categorical Features

In [ ]:
cat_cols = ['territory', 'courier_flow', 'geo_archetype', 'merchant_surface']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(20, 4))
for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
    for p in ax.patches:
        ax.annotate(
            f'{p.get_height() / len(df) * 100:.1f}%',
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha='center', va='bottom', fontsize=8,
        )

plt.suptitle('Categorical Feature Distributions', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
p99_atd = df['ATD'].quantile(0.99)
df_clean = df[df['ATD'] <= p99_atd].copy()

fig, axes = plt.subplots(1, len(cat_cols), figsize=(22, 5))
for ax, col in zip(axes, cat_cols):
    order = df_clean.groupby(col, observed=True)['ATD'].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_clean.sample(n=min(50_000, len(df_clean)), random_state=SEED),
        x=col, y='ATD', order=order, ax=ax,
        color='steelblue',
        flierprops=dict(marker='.', markersize=1, alpha=0.3),
    )
    ax.set_title(f'ATD by {col.replace("_", " ").title()}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)

plt.suptitle('ATD Distribution by Categorical Feature (≤ p99)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
pivot = (
    df_clean
    .groupby(['territory', 'courier_flow'], observed=True)['ATD']
    .agg(['median', 'mean', 'count'])
    .round(1)
    .rename(columns={'median': 'ATD_median', 'mean': 'ATD_mean', 'count': 'trips'})
    .reset_index()
    .sort_values('ATD_median', ascending=False)
)
display(pivot.head(20))

---
## 5. Numerical Features — Distances

In [ ]:
num_cols = ['pickup_distance', 'dropoff_distance']
df_num = df[num_cols + ['ATD']].dropna()

print(f'Rows after dropping distance NAs: {len(df_num):,}')
display(df_num.describe().round(3))

In [ ]:
p99_pick = df_num['pickup_distance'].quantile(0.99)
p99_drop = df_num['dropoff_distance'].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_num.loc[df_num['pickup_distance'] <= p99_pick, 'pickup_distance'],
             bins=60, color='teal', edgecolor='none')
axes[0].set_title(f'Pickup Distance (≤p99={p99_pick:.1f} km)')
axes[0].set_xlabel('km')

axes[1].hist(df_num.loc[df_num['dropoff_distance'] <= p99_drop, 'dropoff_distance'],
             bins=60, color='coral', edgecolor='none')
axes[1].set_title(f'Dropoff Distance (≤p99={p99_drop:.1f} km)')
axes[1].set_xlabel('km')

plt.suptitle('Distance Distributions', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter distances vs ATD
sample = df_num[
    (df_num['ATD'] <= df_num['ATD'].quantile(0.99)) &
    (df_num['pickup_distance'] <= p99_pick) &
    (df_num['dropoff_distance'] <= p99_drop)
].sample(n=min(30_000, len(df_num)), random_state=SEED)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(sample['pickup_distance'], sample['ATD'], alpha=0.05, s=5, color='teal')
axes[0].set_xlabel('Pickup Distance (km)')
axes[0].set_ylabel('ATD (min)')
axes[0].set_title('Pickup Distance vs ATD')

axes[1].scatter(sample['dropoff_distance'], sample['ATD'], alpha=0.05, s=5, color='coral')
axes[1].set_xlabel('Dropoff Distance (km)')
axes[1].set_ylabel('ATD (min)')
axes[1].set_title('Dropoff Distance vs ATD')

plt.suptitle('Distances vs ATD', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Pearson r — pickup_distance  vs ATD: {sample['pickup_distance'].corr(sample['ATD']):.3f}")
print(f"Pearson r — dropoff_distance vs ATD: {sample['dropoff_distance'].corr(sample['ATD']):.3f}")

In [ ]:
df_num = df_num.assign(total_distance=df_num['pickup_distance'] + df_num['dropoff_distance'])
bins   = [0, 1, 2, 3, 5, 8, 15, np.inf]
labels = ['0-1', '1-2', '2-3', '3-5', '5-8', '8-15', '15+']
df_num['total_dist_bin'] = pd.cut(df_num['total_distance'], bins=bins, labels=labels)

atd_by_dist = (
    df_num.groupby('total_dist_bin', observed=True)['ATD']
    .agg(['median', 'mean', 'count']).round(1)
)
display(atd_by_dist)

atd_by_dist['median'].plot(kind='bar', color='steelblue', edgecolor='none', figsize=(10, 4))
plt.title('Median ATD by Total Distance Bucket')
plt.xlabel('Total Distance (km)')
plt.ylabel('Median ATD (min)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Time-Based Patterns

In [ ]:
df_time = df.copy()
df_time['hour']        = df_time['restaurant_offered_timestamp_utc'].dt.hour
df_time['day_of_week'] = df_time['restaurant_offered_timestamp_utc'].dt.day_name()
df_time['date']        = df_time['restaurant_offered_timestamp_utc'].dt.date
df_time['dow_num']     = df_time['restaurant_offered_timestamp_utc'].dt.dayofweek
DOW_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

df_time.groupby('hour').size().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Trip Volume by Hour of Day')
axes[0].set_xlabel('Hour (UTC)')
axes[0].tick_params(axis='x', rotation=0)

df_time.groupby('hour')['ATD'].median().plot(
    kind='line', ax=axes[1], color='coral', marker='o', linewidth=2)
axes[1].set_title('Median ATD by Hour of Day')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Median ATD (min)')

plt.suptitle('Hourly Patterns', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

dow_vol = df_time.groupby('day_of_week').size().reindex(DOW_ORDER)
dow_vol.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Trip Volume by Day of Week')
axes[0].tick_params(axis='x', rotation=35)

atd_dow = df_time.groupby('day_of_week')['ATD'].median().reindex(DOW_ORDER)
atd_dow.plot(kind='bar', ax=axes[1], color='coral', edgecolor='none')
axes[1].set_title('Median ATD by Day of Week')
axes[1].set_ylabel('Median ATD (min)')
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle('Day-of-Week Patterns', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
daily_vol = df_time.groupby('date').size().reset_index(name='trips')
daily_vol['date'] = pd.to_datetime(daily_vol['date'])

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(daily_vol['date'], daily_vol['trips'], color='steelblue', linewidth=1.2)
ax.fill_between(daily_vol['date'], daily_vol['trips'], alpha=0.2, color='steelblue')
ax.set_title('Daily Trip Volume Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Trips')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = (
    df_time.groupby(['dow_num', 'hour'])['ATD']
    .median()
    .unstack('hour')
)
heatmap_data.index = DOW_ORDER

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    heatmap_data, ax=ax, cmap='YlOrRd',
    cbar_kws={'label': 'Median ATD (min)'},
    linewidths=0.3,
)
ax.set_title('Median ATD — Day of Week × Hour (UTC)', fontweight='bold')
ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

---
## 7. Relationships & Correlations

In [ ]:
numeric_df = df[['pickup_distance', 'dropoff_distance', 'ATD']].dropna()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(
    numeric_df.corr(), ax=ax,
    annot=True, fmt='.3f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, vmin=-1, vmax=1,
)
ax.set_title('Pearson Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
p99_atd = df['ATD'].quantile(0.99)
agg_ter = (
    df[df['ATD'] <= p99_atd]
    .groupby(['territory', 'courier_flow'], observed=True)['ATD']
    .median().unstack('courier_flow').fillna(0)
)
agg_ter.plot(kind='bar', figsize=(14, 5), edgecolor='none')
plt.title('Median ATD by Territory & Courier Flow', fontweight='bold')
plt.xlabel('Territory')
plt.ylabel('Median ATD (min)')
plt.xticks(rotation=0)
plt.legend(title='Courier Flow', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

In [ ]:
agg_geo = (
    df[df['ATD'] <= p99_atd]
    .groupby(['geo_archetype', 'merchant_surface'], observed=True)['ATD']
    .median().unstack('merchant_surface').fillna(0)
)
agg_geo.plot(kind='bar', figsize=(14, 5), edgecolor='none')
plt.title('Median ATD by Geo Archetype & Merchant Surface', fontweight='bold')
plt.xlabel('Geo Archetype')
plt.ylabel('Median ATD (min)')
plt.xticks(rotation=20)
plt.legend(title='Merchant Surface', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

---
## 8. Missing Values & Data Quality

In [ ]:
missing_full = (
    df.isnull().sum().rename('missing_count').to_frame()
    .assign(pct=lambda x: (x['missing_count'] / len(df) * 100).round(3))
)
display(missing_full.sort_values('missing_count', ascending=False))

In [ ]:
# Are distance NAs concentrated in a segment?
df['_has_dist'] = df['pickup_distance'].notna()
for col in ['territory', 'courier_flow', 'merchant_surface']:
    pct_missing = (
        df.groupby(col, observed=True)['_has_dist']
        .apply(lambda x: (~x).mean() * 100).round(2)
        .sort_values(ascending=False)
        .rename('missing_%')
    )
    print(f'Distance missing % by {col}:')
    display(pct_missing)
df.drop(columns=['_has_dist'], inplace=True)

In [ ]:
print('ATD sanity checks:')
print(f'  ATD < 0   : {(df["ATD"] < 0).sum():,}')
print(f'  ATD == 0  : {(df["ATD"] == 0).sum():,}')
print(f'  ATD < 1   : {(df["ATD"] < 1).sum():,}')
print(f'  ATD > 120 : {(df["ATD"] > 120).sum():,}')
print(f'  ATD > 240 : {(df["ATD"] > 240).sum():,}')
print(f'  ATD > 480 : {(df["ATD"] > 480).sum():,}')

---
## 9. Key Insights

In [ ]:
p99_atd = df['ATD'].quantile(0.99)

insights = {
    'Total trips': f"{len(df):,}",
    'Date range': (
        f"{df['restaurant_offered_timestamp_utc'].min().date()} – "
        f"{df['restaurant_offered_timestamp_utc'].max().date()}"
    ),
    'ATD median': f"{df['ATD'].median():.1f} min",
    'ATD mean': f"{df['ATD'].mean():.1f} min",
    'ATD p95': f"{df['ATD'].quantile(0.95):.1f} min",
    'ATD p99': f"{df['ATD'].quantile(0.99):.1f} min",
    'Outliers ATD > 120 min': f"{(df['ATD'] > 120).mean() * 100:.1f}%",
    'Top territory (volume)': str(df['territory'].value_counts().index[0]),
    'Top courier_flow': str(df['courier_flow'].value_counts().index[0]),
    'Top geo_archetype': str(df['geo_archetype'].value_counts().index[0]),
    'Top merchant_surface': str(df['merchant_surface'].value_counts().index[0]),
    'Missing pickup_distance': f"{df['pickup_distance'].isnull().mean() * 100:.2f}%",
    'Missing driver_uuid': f"{df['driver_uuid'].isnull().mean() * 100:.2f}%",
    'Corr dropoff_distance ~ ATD': f"{df[['dropoff_distance', 'ATD']].dropna().corr().iloc[0,1]:.3f}",
}
display(pd.Series(insights, name='value').to_frame())

### Summary Table

| # | Finding | Implication |
|---|---------|-------------|
| 1 | **Median ATD ≈ 36 min**, mean ≈ 47 min — right-skewed distribution | Use median as primary KPI; cap or model outliers separately |
| 2 | **~1–2% of trips have ATD > 120 min** (up to 1 911 min) | Remove or Winsorize extreme outliers before training |
| 3 | **Motorbike dominates** (~79% of trips) | Stratify by `courier_flow`; the minority classes may behave very differently |
| 4 | **Peak hours: 19–23 h and 0–2 h UTC** (dinner + late-night in MX local time) | Include hour-of-day and is_peak_hour as model features |
| 5 | **Dropoff distance correlates more strongly with ATD** than pickup distance | Prioritise total distance & dropoff in feature engineering |
| 6 | **~2.5% missing distances** — not uniformly distributed across segments | Impute with territory × courier_flow median; add `is_distance_missing` flag |
| 7 | **Central territory has the highest trip volume** | Consider territory-level bias correction or separate models |
| 8 | **POS merchant surface is most common but shows different ATD** vs Tablet | `merchant_surface` is a meaningful feature for preparation time |
